# TP2 - Etapa 4 en Google Colab

Evaluacion del pipeline completo, optimizacion de modelos y anotacion automatica.

Esta etapa se trabaja en Colab para aprovechar la GPU en la optimizacion y la inferencia:
**Entorno de ejecucion -> Cambiar tipo de entorno de ejecucion -> GPU (T4)**.

Requisitos previos (en tu fork):
- Etapas 1 a 3 implementadas (`evaluate_pipeline` y `generate_annotations` reutilizan
  `detect_dogs` y `classify_detected_dog`).
- Funciones de la Etapa 4 implementadas en `src/lib/services/pipeline_service.py`.
- Conjunto de prueba versionado en `data/eval` (al menos 10 imagenes complejas anotadas
  manualmente con bounding boxes y raza).
- Checkpoints entrenados disponibles en un link publico de solo lectura (no se versionan).

## 1. Clonar el repositorio

Si tu fork es privado, genera un token de acceso (GitHub -> Settings -> Developer settings ->
Personal access tokens) y usalo en la URL, o sube un zip del proyecto a Colab/Drive.

In [ ]:
# TO-DO: completar con la URL de tu fork.
# Repo privado: https://<TOKEN>@github.com/<usuario>/tuia-dog-recognition-app.git
REPO_URL = "https://github.com/<usuario>/tuia-dog-recognition-app.git"

!git clone $REPO_URL proyecto
%cd proyecto

## 2. Instalar dependencias

Colab ya incluye torch, torchvision, opencv, numpy, scikit-learn y matplotlib;
solo se instala lo que falta.

In [ ]:
!pip install -q pydantic-settings python-dotenv ultralytics onnx onnxruntime

## 3. Configuracion del entorno

En Colab no hay PostgreSQL; la Etapa 4 no usa la base vectorial, por lo que se desactiva
pgvector. La configuracion se define por variables de entorno **antes** de importar `lib`.

In [ ]:
import os
import sys
from pathlib import Path

os.environ["USE_PGVECTOR"] = "false"
# Ajusta aca cualquier otra variable (YOLO_MODEL, YOLO_CONF_THRESHOLD, IMAGE_SIZE, etc.)

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from lib.config import settings

print("eval_path:", settings.eval_path)
print("yolo_model:", settings.yolo_model)

## 4. Descargar los checkpoints entrenados

Los modelos de la Etapa 2 no se versionan en git: descargalos a `models/` desde tu link
publico (Drive, Hugging Face, etc.).

In [ ]:
# TO-DO: descargar los checkpoints a models/.
# Ejemplo con gdown (archivo en Google Drive):
#   !pip install -q gdown
#   !gdown <ID_DEL_ARCHIVO> -O models/resnet18_finetuned.pth
# Ejemplo con un link directo:
#   !wget -q <URL> -O models/resnet18_finetuned.pth

!ls -lh models/

## 5. Construir los servicios

In [ ]:
from lib.bootstrap import build_classifier, build_detection, build_pipeline

classifier = build_classifier(settings)
detection = build_detection(settings, classifier)
pipeline = build_pipeline(settings, detection)

## 6. Evaluacion del pipeline completo

Calcula mAP, IoU, precision, recall y F1 sobre `data/eval`
(helpers en `lib.evaluation.metrics`).

In [ ]:
metrics = pipeline.evaluate_pipeline()
metrics

### Analisis de resultados

TO-DO: analisis detallado de las metricas obtenidas, incluyendo falsos positivos y
falsos negativos (con ejemplos visuales usando `lib.visualization.draw.draw_detections`).

In [ ]:
## TO-DO: visualizacion y analisis de errores.

## 7. Optimizacion de modelos

Una estrategia a eleccion: cuantizacion (FP32 -> INT8) o exportacion optimizada
(ONNX / TensorRT). Comparar tiempo de inferencia, uso de memoria y precision.

In [ ]:
report = pipeline.optimize_model()
report

## 8. Anotacion automatica

Genera anotaciones en formato YOLOv5 y COCO para una carpeta de imagenes.

In [ ]:
yolo_out = pipeline.generate_annotations("data/eval", "yolo")
coco_out = pipeline.generate_annotations("data/eval", "coco")
print(yolo_out)
print(coco_out)

In [ ]:
# Descargar los resultados generados en output/ (anotaciones, modelos optimizados, etc.)
import shutil

shutil.make_archive("etapa4_resultados", "zip", "output")
try:
    from google.colab import files

    files.download("etapa4_resultados.zip")
except ImportError:
    print("Fuera de Colab: el zip quedo en ./etapa4_resultados.zip")

## 9. Cierre

TO-DO:
- Copiar las metricas y la comparacion de optimizacion al informe (`informe.ipynb`).
- Commitear en el fork los cambios de `pipeline_service.py` y las anotaciones generadas
  que se quieran entregar (recordar que `output/` esta gitignoreado: mover lo relevante
  a una carpeta versionada si corresponde).